# 21 — Simple Agent1 chat
Use the code below in notebook 16's loaded kernel. A new notebook does not automatically share its model. No extra web framework or server is needed: this is a notebook/terminal text interface.

1. Paste the setup below into notebook 16.
2. In the next cell run `run_chat(session)`.
3. Type messages at the input prompt. Type `/quit` to release the cell.

Alternatively, configure your ordinary Agent in this notebook once, then run the setup. Do not load a second copy of the model while another notebook already occupies GPU memory.

```python
# Paste in notebook 16 AFTER the final source-backed Agent setup.
# Reuses its model and classifier; no restart, model load or wrapper is needed.
from pathlib import Path
from agent1.persistent_agent import PersistentAgent
from agent1.session import ChatSession
from agent1.chat_console import run_chat
from agent1.persistence.sqlite_session_store import SQLiteSessionStore

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src/agent1").is_dir())
template_agent = agent
store = SQLiteSessionStore(project_root / "data/local/conversations.sqlite3")

# Optional MongoDB replacement after notebook 20 passes:
# import os
# from agent1.persistence.mongo_store import MongoCheckpointStore
# store = MongoCheckpointStore.from_uri(os.environ["MONGODB_URI"], database="agent1_demo")

def make_agent(checkpoint_store):
    return PersistentAgent(
        ensemble_model=template_agent.ensemble_model,
        router=template_agent.router,
        analyzer=template_agent.analyzer,
        context_builder=template_agent.context_builder,
        top_n_intents=template_agent.top_n_intents,
        response_service=template_agent.response_service,
        checkpoint_store=checkpoint_store,
    )

session = ChatSession(store, make_agent)
print("Session interface ready; model reused.")
# In the next cell, run:
# run_chat(session)

```

## Commands
| Command | Behavior |
|---|---|
| `/new` | Start and save an empty conversation |
| `/list` | Show up to 20 saved conversations |
| `/open ID` | Open that saved conversation and show its messages |
| `/history` | Show user/assistant messages for the active conversation |
| `/retry-save` | Retry only persistence after a save failure |
| `/reload` | Confirm discarding unsaved progress and restore the last saved checkpoint |
| `/quit` | Exit the chat loop |

Ordinary messages call the existing agent and save after the turn. Start with `Track my order and check whether the blue backpack is available.` Then use `/quit`, call `run_chat(session)` again, and provide `ORD-123`. To test a fresh session object, run `session = ChatSession(store, make_agent)`, then `session.open("your saved ID")`, then `run_chat(session)`.

A save error labels the answer NOT CONFIRMED SAVED and blocks further turns until recovery. `/retry-save` does not repeat tools or inference. Processing errors require `/reload` to discard the unsaved turn. Loading an old pending checkpoint may repeat read-only lookups if you subsequently resume it. This is not exactly-once crash recovery.

This interface completes the planned minimal interaction layer. Model planning limitations remain; the persistence layer does not correct semantic errors.
